# Business Entity Resolution Challenge - Interactive ML Pipeline

This notebook implements an end-to-end Machine Learning pipeline for Business Entity Resolution using:
- **pandas** & **scikit-learn**
- **TF-IDF Sparse Candidate Blocking** (Stage 1)
- **Pairwise Similarity Feature Engineering** (Levenshtein, Jaro-Winkler, Jaccard, Token Overlap, Open-set Country match) (Stage 2)
- **LightGBM Binary Classifier** & **Macro $F_{0.5}$ Threshold Tuning** (Stage 3)
- **Submission Output Generation & Format Validation** (Stage 4)

## 1. Imports and Setup

In [1]:
import os
import sys
import pandas as pd

# Add project root directory to path
sys.path.append(os.getcwd())

from src.preprocess import load_source_file, preprocess_dataframe, normalize_text
from src.blocking import CandidateBlocker
from src.feature_engineering import PairFeatureExtractor
from src.train import train_entity_resolution_model
from src.inference import run_test_inference
from utils.validate_submission import validate_submission
from utils.generate_sample_data import create_dataset

print("Project modules imported successfully!")

Project modules imported successfully!


## 2. Ingest Data & Inspect Dataset

In [2]:
# Generate dataset if missing
if not (os.path.exists("dataset/train/train_source1.tsv") and os.path.exists("dataset/test/test_source1.tsv")):
    create_dataset("train", 150)
    create_dataset("test", 100)

df_s1 = load_source_file("dataset/train/train_source1.tsv")
df_s2 = load_source_file("dataset/train/train_source2.tsv")
df_s3 = load_source_file("dataset/train/train_source3.tsv")

print(f"Source 1 Records: {len(df_s1)}")
print(f"Source 2 Records: {len(df_s2)}")
print(f"Source 3 Records: {len(df_s3)}")
df_s1.head()

Source 1 Records: 150
Source 2 Records: 113
Source 3 Records: 106


,entity_id,business_name,business_address,country
0,S1-00001,Horizon Industrial Machinery 1,"707 Boulevard Haussmann, Paris, France",US
1,S1-00002,Beacon Retail Outlets Company 2,"505 Ring Road, Anna Nagar, Chennai, TN",US
2,S1-00003,Horizon Industrial Machinery 3,"303 Industrial Layout, Phase 1, Hyderabad, TS",US
3,S1-00004,Nexus Digital Media Agency 4,"404 Broadway Avenue, Suite 10, Chicago, IL",India
4,S1-00005,Metro Urban Infrastructure 5,"12 Main Street, Suite 400, New York, NY",US


## 3. Stage 1: Candidate Generation (Blocking)

In [3]:
df_s1_prep = preprocess_dataframe(df_s1)
df_s2_prep = preprocess_dataframe(df_s2)
df_s3_prep = preprocess_dataframe(df_s3)

blocker = CandidateBlocker(top_k=50)
candidate_map = blocker.fit_transform_candidates(df_s1_prep, df_s2_prep, df_s3_prep)
print("Total S1 entities blocked:", len(candidate_map))
print("Sample candidate list for S1-00001:", candidate_map.get("S1-00001", [])[:5])

Total S1 entities blocked: 150
Sample candidate list for S1-00001: ['S2-00001', 'S3-00059', 'S3-00072', 'S3-00066', 'S2-00061']


## 4. Stage 2 & 3: Model Training & Macro $F_{0.5}$ Threshold Optimization

In [4]:
artifacts = train_entity_resolution_model(
    train_dir="dataset/train",
    model_output_path="models/er_model.pkl",
    model_type="lightgbm"
)

print(f"Optimal Threshold: {artifacts['optimal_threshold']:.2f}")
print(f"Validation Macro F0.5 Score: {artifacts['best_val_f05']:.4f}")

=== Starting Model Training & Optimization Pipeline ===
Running Candidate Generation (Blocking)...
Constructing Candidate Pairwise Dataset...
Total candidate pairs generated: 7500 (Positive matches: 179)
Extracting Similarity Features...
Training LIGHTGBM Classifier...
Optimizing Classification Probability Threshold for Macro F_0.5...

[RESULTS] Validation Macro F_0.5 Score: 0.9649 at Optimal Threshold: 0.35
Model and artifacts successfully saved to models/er_model.pkl
Optimal Threshold: 0.35
Validation Macro F0.5 Score: 0.9649


## 5. Stage 4: Test Set Inference & Output Generation

In [5]:
run_test_inference(
    test_dir="dataset/test",
    model_path="models/er_model.pkl",
    candidate_output_path="output/candidate_pairs.tsv",
    matching_output_path="output/matching_results.tsv"
)

=== Starting Test Set Inference & Submission Generation ===
Loaded model from models/er_model.pkl. Using optimal classification threshold: 0.35
Generating Candidate Pairs for Test Set...
Saved 100 blocking candidate rows to output/candidate_pairs.tsv
Extracting Test Pair Similarity Features...
Predicting Match Probabilities with Model...
Successfully generated 100 matching result rows in output/matching_results.tsv
=== Submitting Submission Format & Rule Validator ===

[PASS] Validation successful! Submission files are 100% compliant with all rules.


## 6. Submission Format & Constraint Validation

In [6]:
is_valid = validate_submission(
    matching_path="output/matching_results.tsv",
    candidate_path="output/candidate_pairs.tsv",
    test_dir="dataset/test"
)
print("Submission format fully valid?", is_valid)

=== Submitting Submission Format & Rule Validator ===

[PASS] Validation successful! Submission files are 100% compliant with all rules.
Submission format fully valid? True
